In [1]:
import sys, time
sys.path.append("..")
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.retrieval import make_dense_ranker
from src.metrics import evaluate_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
ranker = make_dense_ranker()

c = contracts[dev_idx[0]]
top = ranker(c, "Governing Law")[:3]
print(c["title"], "| gold:", c["gold"]["Governing Law"])
for ch in top:
    print(round(ch["score"], 3), ch["start"], ch["end"])
    print(ch["text"][:300].replace("\n", " "))
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT | gold: [(52061, 52151)]
0.582 48000 49000
of said                   provisions,   rights  or  elections   shall  not  preclude  or                   prejudice  such party from later  enforcing or exercising  the                   same or any other provisions, rights or elections which is may                   have under this Agreement.     

0.579 43200 44200
validity or  enforceability  of any                   patent,  copyright,  trademark,  trade name or other propriety                   right owned by or licensed to the Company,  whether  currently                   existing or hereinafter invented, developed or acquired unless                   req

0.579 53600 54290
erein provided,  understanding  that such terms                   as price,  quotas,  and  length  of the term of the  agreement                   shall be  reasonably  adjusted  to  reflect  the nature of the                   other Product or device which is the 

In [2]:
t = time.time()
evaluate_ranker(ranker, contracts, dev_idx[:20], "Governing Law")
print(round(time.time() - t, 1), "seconds for 20 contracts")

1.4 seconds for 20 contracts


In [3]:
rows = {cl: evaluate_ranker(ranker, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
dense = pd.DataFrame(rows).T
dense.to_csv("../eval/results_dense_fixed1000_dev.csv")
dense

,hit@1,hit@3,hit@5,hit@10,span_recall@1,span_recall@3,span_recall@5,span_recall@10,mrr,n
Governing Law,0.585,0.837,0.889,0.944,0.574,0.827,0.882,0.939,0.723,306.0
Renewal Term,0.712,0.856,0.924,0.975,0.712,0.847,0.914,0.970,0.794,118.0
Notice Period To Terminate Renewal,0.690,0.859,0.901,0.958,0.671,0.852,0.894,0.951,0.784,71.0
Termination For Convenience,0.407,0.669,0.746,0.898,0.367,0.633,0.718,0.872,0.567,118.0
Non-Compete,0.280,0.467,0.600,0.800,0.207,0.352,0.467,0.653,0.428,75.0
Cap On Liability,0.621,0.879,0.926,0.963,0.467,0.727,0.807,0.888,0.756,190.0
Uncapped Liability,0.667,0.840,0.951,1.000,0.588,0.805,0.927,0.987,0.781,81.0
